# Random forests: many trees voting

A gym wants to call members who are about to cancel. Its one deep decision tree, retrained each month, gives
many members a different verdict every time. In this notebook you build the same invented gym members as the
lesson, compare one tree with a forest, vote with three small trees by hand, turn the three knobs that matter,
and catch an importance chart that ranks a random number.

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no key and no download.

The members are invented by a small program with fixed seeds. They are not real people.

## 1. Make the gym members

The next cell holds the small program that invents the members. Each member has visits per week, contract plan
(0 = monthly, 1 = annual), months as a member, days since the last visit, age, and a member id (a random
six-digit number). `cancelled` is 1 if the member cancelled the next month. The label is drawn at random from a
chance, so two similar members can end differently: that is the noise a deep tree memorises.

It makes three groups with fixed seeds: January (400 members), February (400) and a validation set (1,000)
that no model trains on. Most models use the first five columns only; `member_id` comes back in step 8.

**Look for:** 400, 400 and 1,000 rows. Cancel rates of 0.318, 0.287 and 0.286.

In [1]:
import numpy as np
import sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

print("scikit-learn version:", sklearn.__version__)

FEATURES = ["visits_per_week", "annual_contract", "tenure_months", "days_since_last_visit", "age", "member_id"]


def make_members(n, seed):
    rng = np.random.default_rng(seed)
    visits = np.round(rng.gamma(2.0, 1.0, n).clip(0, 6), 1)
    annual = (rng.random(n) < 0.4).astype(int)
    tenure = rng.integers(1, 49, n)
    days = rng.integers(0, 31, n)
    age = rng.integers(18, 71, n)
    member_id = rng.integers(100000, 1000000, n)
    logit = 0.4 - 0.7 * visits - 1.3 * annual - 0.03 * tenure + 0.07 * days
    p = 1 / (1 + np.exp(-logit))
    cancelled = (rng.random(n) < p).astype(int)
    X = np.column_stack([visits, annual, tenure, days, age, member_id])
    return X, cancelled


# seeds: January 1, February 2, validation 3
X_jan_all, y_jan = make_members(400, 1)
X_feb_all, y_feb = make_members(400, 2)
X_val_all, y_val = make_members(1000, 3)

COLS = [0, 1, 2, 3, 4]  # every feature except member_id
names = [FEATURES[c] for c in COLS]
X_jan, X_feb, X_val = X_jan_all[:, COLS], X_feb_all[:, COLS], X_val_all[:, COLS]

print("rows: january", len(y_jan), " february", len(y_feb), " validation", len(y_val))
print("cancel rate: january %.3f  february %.3f  validation %.3f" % (y_jan.mean(), y_feb.mean(), y_val.mean()))

scikit-learn version: 1.9.1
rows: january 400  february 400  validation 1000
cancel rate: january 0.318  february 0.287  validation 0.286


## 2. One deep tree, trained twice

This is what the gym does now: one decision tree with no depth limit, grown on January, and again on February.
Each is scored on its own training rows and on the 1,000 validation members.

**Look for:** both trees score 1.000 on their training rows but only 0.656 (January) and 0.693 (February) on
validation. That gap is the accidents each tree memorised. Both start with almost the same first question:
`visits_per_week <= 1.7`, then `<= 1.6`.

In [2]:
def first_question(tree):
    t = tree.tree_
    return "%s <= %.1f" % (names[t.feature[0]], t.threshold[0])

tree_jan = DecisionTreeClassifier(random_state=0).fit(X_jan, y_jan)
tree_feb = DecisionTreeClassifier(random_state=0).fit(X_feb, y_feb)
for month, m, X, y in [("january", tree_jan, X_jan, y_jan), ("february", tree_feb, X_feb, y_feb)]:
    print("tree %-8s train acc %.3f  validation acc %.3f  depth %d  leaves %d  first question: %s" % (
        month, m.score(X, y), m.score(X_val, y_val), m.get_depth(), m.get_n_leaves(), first_question(m)))

tree january  train acc 1.000  validation acc 0.656  depth 16  leaves 106  first question: visits_per_week <= 1.7
tree february train acc 1.000  validation acc 0.693  depth 12  leaves 99  first question: visits_per_week <= 1.6


## 3. The same members, a different verdict

The next cell asks both trees about the validation members and counts the "flips": members whose verdict
changes between the January tree and the February tree. It also prints the first eight members, as on the
lesson's first screen.

**Look for:** 343 of 1,000 members flip. Among the first eight, members 5 and 7 flip, as on the lesson's
first screen.

In [3]:
verdict = lambda v: "cancel" if v else "stay"
tree_pj, tree_pf = tree_jan.predict(X_val), tree_feb.predict(X_val)
print("tree flips january -> february:", int((tree_pj != tree_pf).sum()), "of", len(y_val))
print()
print("#  visits  plan     months  days away  age  | Jan tree  Feb tree")
for i in range(8):
    v, a, t, d, age = X_val[i]
    mark = "  <- flips" if tree_pj[i] != tree_pf[i] else ""
    print("%d  %4.1f    %-7s  %4d    %4d       %3d  | %-8s  %-8s%s" % (
        i + 1, v, "annual" if a else "monthly", t, d, age, verdict(tree_pj[i]), verdict(tree_pf[i]), mark))

tree flips january -> february: 343 of 1000

#  visits  plan     months  days away  age  | Jan tree  Feb tree
1   5.9    monthly    19      30        60  | stay      stay    
2   2.3    annual     43      20        50  | stay      stay    
3   1.1    annual     44      28        39  | stay      stay    
4   0.2    monthly    31      19        53  | stay      stay    
5   0.8    annual     41      24        70  | cancel    stay      <- flips
6   2.0    monthly    12       0        22  | stay      stay    
7   1.3    monthly    26       0        70  | stay      cancel    <- flips
8   0.6    annual     14       3        31  | stay      stay    


## 4. Three small trees vote, by hand

Before the forest, the idea in small. The cell takes ten January members (the first five who cancelled and the
first five who stayed), named A to J, and three of their columns. For each tree it draws ten rows **at random,
with repeats**: a bootstrap sample. Then it grows a tree with one question on each sample and asks it about a
new member: 1.4 visits a week, monthly plan, last visit 12 days ago.

The cell tries bootstrap seeds from 0 upwards and keeps the first one whose three trees do not all agree, so
the vote is worth looking at.

**Look for:** seed 1. Tree 1 votes stay, tree 2 votes cancel, tree 3 votes stay, so the majority is stay.
Tree 2's sample has no B, D or H: it never saw a low-visit member who stayed.

In [4]:
BY_COLS = [0, 1, 3]
by_names = [FEATURES[c] for c in BY_COLS]
idx = sorted(list(np.where(y_jan == 1)[0][:5]) + list(np.where(y_jan == 0)[0][:5]))
X10, y10 = X_jan_all[idx][:, BY_COLS], y_jan[idx]
letters = "ABCDEFGHIJ"
NEW = np.array([[1.4, 0, 12]])  # the new member: 1.4 visits a week, monthly contract, last visit 12 days ago

def attempt(seed):
    rng = np.random.default_rng(seed)
    out = []
    for t in range(3):
        rows = np.sort(rng.integers(0, 10, 10))
        tree = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X10[rows], y10[rows])
        f, thr = tree.tree_.feature[0], tree.tree_.threshold[0]
        if f < 0:
            return None
        side = NEW[0, f] <= thr
        lab = y10[rows][(X10[rows, f] <= thr) == side]
        if lab.sum() * 2 == len(lab):
            return None
        out.append((rows, by_names[f], thr, bool(side), int(lab.sum()), int(len(lab) - lab.sum()), int(tree.predict(NEW)[0])))
    votes = [o[-1] for o in out]
    if len(set(o[1] for o in out)) < 2 or sum(votes) not in (1, 2):
        return None
    return out

for seed in range(1000):
    res = attempt(seed)
    if res:
        break
print("bootstrap seed", seed)
print("ten members (letter, visits_per_week, annual_contract, days_since_last_visit, cancelled):")
for L, r, c in zip(letters, X10, y10):
    print(" ", L, r[0], int(r[1]), int(r[2]), int(c))
for t, (rows, fname, thr, side, n_cancel, n_stay, vote) in enumerate(res, 1):
    print("tree %d sample: %s" % (t, " ".join(letters[r] for r in rows)))
    print("  question: %s <= %.2f   new member answers %s" % (fname, thr, "yes" if side else "no"))
    print("  sample rows on that side: cancelled %d, stayed %d -> vote %s" % (n_cancel, n_stay, verdict(vote)))
votes = [o[-1] for o in res]
print("votes:", votes, " majority:", verdict(sum(votes) >= 2))

bootstrap seed 1
ten members (letter, visits_per_week, annual_contract, days_since_last_visit, cancelled):
  A 2.2 0 4 0
  B 2.1 0 14 0
  C 3.1 1 20 0
  D 1.1 0 7 0
  E 2.2 0 11 0
  F 4.0 1 20 1
  G 0.3 1 19 1
  H 0.9 0 24 1
  I 0.8 1 15 1
  J 0.6 0 19 1
tree 1 sample: A B C D E F H I J J
  question: visits_per_week <= 1.00   new member answers no
  sample rows on that side: cancelled 1, stayed 5 -> vote stay
tree 2 sample: A A C C E E F G I I
  question: visits_per_week <= 1.50   new member answers yes
  sample rows on that side: cancelled 3, stayed 0 -> vote cancel
tree 3 sample: B D D E F H H I I I
  question: days_since_last_visit <= 14.50   new member answers yes
  sample rows on that side: cancelled 0, stayed 4 -> vote stay
votes: [0, 1, 0]  majority: stay


## 5. A forest of 100 trees, trained twice

`RandomForestClassifier` does the same at scale: 100 deep trees, each on its own bootstrap sample, each split
looking at a few random features. The cell trains it on January and on February, like the trees in step 2,
and counts flips again.

**Look for:** validation accuracy 0.729 (January) and 0.735 (February), against 0.656 and 0.693 for the single
tree. Training accuracy is 0.998, so training accuracy tells you nothing here. Flips fall from 343 to 192:
steadier, not still.

In [5]:
forest_jan = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_jan, y_jan)
forest_feb = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_feb, y_feb)
for month, m, X, y in [("january", forest_jan, X_jan, y_jan), ("february", forest_feb, X_feb, y_feb)]:
    print("forest %-8s train acc %.3f  validation acc %.3f" % (month, m.score(X, y), m.score(X_val, y_val)))

forest_flips = int((forest_jan.predict(X_val) != forest_feb.predict(X_val)).sum())
print("tree flips january -> february:  ", int((tree_pj != tree_pf).sum()), "of", len(y_val))
print("forest flips january -> february:", forest_flips, "of", len(y_val))

forest january  train acc 0.998  validation acc 0.729


forest february train acc 0.998  validation acc 0.735
tree flips january -> february:   343 of 1000
forest flips january -> february: 192 of 1000


## 6. The three knobs

`run` trains a forest with the given settings on January and on February. It prints the January forest's
validation accuracy and the flips between the two. Rows: more trees (`n_estimators`), every feature allowed at
each split (`max_features=None`, which is plain bagging), and at least 5 training rows per leaf
(`min_samples_leaf=5`).

**Look for:** 1 tree 0.608, 10 trees 0.706, 100 trees 0.729, 500 trees 0.726: the gain stops after about 100.
All features per split: 0.710 and 234 flips, worse and jumpier. `min_samples_leaf=5`: 0.750 and 116 flips,
the steadiest. This cell takes a few seconds.

In [6]:
def run(**settings):
    a = RandomForestClassifier(random_state=0, **settings).fit(X_jan, y_jan)
    b = RandomForestClassifier(random_state=0, **settings).fit(X_feb, y_feb)
    flips = int((a.predict(X_val) != b.predict(X_val)).sum())
    print("%-46s validation acc %.3f  flips %3d of %d" % (settings, a.score(X_val, y_val), flips, len(y_val)))

for n in [1, 10, 100, 500]:
    run(n_estimators=n)
run(n_estimators=100, max_features=None)
run(n_estimators=100, min_samples_leaf=5)

{'n_estimators': 1}                            validation acc 0.608  flips 400 of 1000
{'n_estimators': 10}                           validation acc 0.706  flips 235 of 1000


{'n_estimators': 100}                          validation acc 0.729  flips 192 of 1000


{'n_estimators': 500}                          validation acc 0.726  flips 190 of 1000


{'n_estimators': 100, 'max_features': None}    validation acc 0.710  flips 234 of 1000
{'n_estimators': 100, 'min_samples_leaf': 5}   validation acc 0.750  flips 116 of 1000


## 7. The code from the lesson's library screen

The same ten lines as the lesson. `oob_score=True` scores each training row using only the trees whose
bootstrap sample left that row out: a free check, with no extra data. `predict_proba` gives one row per
member: the chance of stay, then the chance of cancel, averaged over the 300 trees.

**Look for:** `oob: 0.725` and `val: 0.749`, close to each other. The first member's row is `[0.66 0.34]`.

In [7]:
members_jan, cancelled_jan = X_jan, y_jan
members_val, cancelled_val = X_val, y_val

forest = RandomForestClassifier(
    n_estimators=300, max_features="sqrt",
    min_samples_leaf=5, oob_score=True,
    n_jobs=-1, random_state=0)
forest.fit(members_jan, cancelled_jan)
print("oob:", round(forest.oob_score_, 3))
print("val:", round(forest.score(
    members_val, cancelled_val), 3))
print(forest.predict_proba(
    members_val[:3]).round(2))

oob: 0.725
val: 0.749
[[0.66 0.34]
 [0.93 0.07]
 [0.69 0.31]]


## 8. Broken on purpose: the chart that blames a member number

The analyst trains a forest on all six columns, `member_id` included by mistake, and prints its built-in
`feature_importances_`. This number adds up how much each column's splits cleaned up the **training** rows.

**Look for:** training accuracy 1.000. `member_id`, a random number, gets 0.188: fourth of six, almost level with
`days_since_last_visit` (0.191). `annual_contract` gets only 0.035: it has two values, and this kind of importance
favours columns with many different values.

In [8]:
broken = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_jan_all, y_jan)
print("train acc %.3f  validation acc %.3f" % (broken.score(X_jan_all, y_jan), broken.score(X_val_all, y_val)))
print("feature_importances_ (from training splits):")
for name, v in sorted(zip(FEATURES, broken.feature_importances_), key=lambda t: -t[1]):
    print("  %-22s %.3f" % (name, v))

train acc 1.000  validation acc 0.719
feature_importances_ (from training splits):
  visits_per_week        0.222
  tenure_months          0.208
  days_since_last_visit  0.191
  member_id              0.188
  age                    0.156
  annual_contract        0.035


## 9. The honest check: permutation importance on validation

`permutation_importance` shuffles one column in the validation rows, 10 times, and measures how much accuracy
drops. A column the model only used to memorise training members drops nothing on new members. Then the cell
trains the forest again without `member_id`.

**Look for:** `visits_per_week` 0.038 and `days_since_last_visit` 0.021 at the top. `age` (-0.009) and
`member_id` (-0.015) come out below zero: shuffling them does not hurt. Without `member_id`, validation accuracy
goes from 0.719 to 0.725.

In [9]:
perm = permutation_importance(broken, X_val_all, y_val, n_repeats=10, random_state=0)
print("permutation importance on validation (mean drop in accuracy, spread):")
for name, m, s in sorted(zip(FEATURES, perm.importances_mean, perm.importances_std), key=lambda t: -t[1]):
    print("  %-22s %6.3f  %.3f" % (name, m, s))

fixed = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_jan, y_jan)
print("without member_id: validation acc %.3f" % fixed.score(X_val, y_val))

permutation importance on validation (mean drop in accuracy, spread):
  visits_per_week         0.038  0.012
  days_since_last_visit   0.021  0.009
  tenure_months           0.003  0.008
  annual_contract         0.001  0.005
  age                    -0.009  0.006
  member_id              -0.015  0.004


without member_id: validation acc 0.725


## Now change it

Each cell below has one line marked `# <- change this`. Change the number, run the cell, and compare.

### Experiment 1: features per split

`max_features` is how many of the 5 features each split may look at, picked at random. `"sqrt"` means 2.

**What should change:** 2 gives 0.729 and 192 flips. With 5 it is plain bagging: every split may pick the
same strong feature, the trees grow alike, and flips go up to 234 while accuracy falls to 0.710 (3 sits between:
0.720 and 215). With 1 the trees are even more different: 159 flips, 0.734 on this data.
The vote can only cancel mistakes that differ between trees.

In [10]:
features_per_split = 2  # <- change this (try 1, 3, 5)
run(n_estimators=100, max_features=features_per_split)

{'n_estimators': 100, 'max_features': 2}       validation acc 0.729  flips 192 of 1000


### Experiment 2: leaf size

`min_samples_leaf` stops any leaf from holding fewer training rows than this number, so a tree can no longer
give one odd member its own leaf.

**What should change:** 1 is the default (0.729, 192 flips). Larger leaves remove noise inside every tree, so
flips fall: 116 at 5, 86 at 10, 76 at 20, 40 at 50. Accuracy rises too, up to a point (0.758 at 10 and 20),
then falls at 50 (0.740): the trees now know too little. Always compare on validation.

In [11]:
leaf_size = 5  # <- change this (try 1, 10, 20, 50)
run(n_estimators=100, min_samples_leaf=leaf_size)

{'n_estimators': 100, 'min_samples_leaf': 5}   validation acc 0.750  flips 116 of 1000


### Experiment 3: how many different values a useless column has

The cell replaces `member_id` with a random number that has a given count of different values, trains the
forest on all six columns, and prints that column's built-in importance and its permutation importance on
validation.

**What should change:** with 900,000 values (like a member id) the useless column gets 0.182, rank 4 of 6.
With 100 values it gets 0.171, with 10 values 0.119, and with 2 values (like a coin flip) 0.035, last of 6.
The column is equally useless every time; only its number of different values changed. Its permutation
importance stays at or just below zero every time.

In [12]:
distinct_values = 900000  # <- change this (try 2, 10, 100)
rng = np.random.default_rng(0)
Xj_noise, Xv_noise = X_jan_all.copy(), X_val_all.copy()
Xj_noise[:, 5] = rng.integers(0, distinct_values, len(Xj_noise))
Xv_noise[:, 5] = rng.integers(0, distinct_values, len(Xv_noise))
f = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xj_noise, y_jan)
p = permutation_importance(f, Xv_noise, y_val, n_repeats=10, random_state=0)
print("distinct values:", distinct_values)
print("built-in importance of the random column: %.3f  (rank %d of 6)" % (
    f.feature_importances_[5], 1 + int((f.feature_importances_ > f.feature_importances_[5]).sum())))
print("permutation importance of the random column: %.3f" % p.importances_mean[5])

distinct values: 900000
built-in importance of the random column: 0.182  (rank 4 of 6)
permutation importance of the random column: -0.003


## What you should see

- Step 2: the single January tree scores 1.000 on its training rows and 0.656 on the validation members.
- Step 5: the 100-tree forest scores 0.729 on validation, higher than the single tree.
- Step 5: 192 validation members flip between the January and February forests, against 343 for the two trees.
- Step 6: going from 100 to 500 trees moves validation accuracy only from 0.729 to 0.726.
- Steps 8 and 9: `feature_importances_` gives `member_id` 0.188, close to the real features; permutation importance on validation puts it below zero (-0.015).